In [1]:
import pandas as pd
import re

In [2]:
shopping_queries_dataset_sources = pd.read_csv("../data/shopping_queries_dataset_sources.csv")
shopping_queries_dataset_products = pd.read_parquet("../data/shopping_queries_dataset_products.parquet")
shopping_queries_dataset_examples = pd.read_parquet("../data/shopping_queries_dataset_examples.parquet")

In [3]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [4]:
electronics_keywords = [
    # mobile & computing
    "phone", "smartphone", "laptop", "notebook computer", "tablet", "desktop computer",
    "smartwatch", "chromebook",
    # components
    "ram", "ddr4", "ddr5", "ssd", "hard drive", "processor", "cpu", "gpu", "motherboard",
    "power bank", "charger", "usb cable", "hdmi cable",
    # imaging
    "camera", "dash cam", "webcam", "security camera", "camcorder",
    # home electronics
    "television", " tv ", "smart speaker", "router", "wifi extender", "bluetooth speaker",
    # audio
    "headphone", "earbud", "earphone"
]

brand_allowlist = [
    "samsung", "apple", "xiaomi", "sony", "lg", "dell", "hp", "lenovo", "asus",
    "acer", "panasonic", "canon", "nikon", "gopro", "logitech", "jbl", "bose",
    "anker", "tp-link", "netgear", "corsair", "kingston", "western digital", "seagate"
]

conglomerate_brands = ['samsung', 'sony', 'lg', 'panasonic']

In [16]:
import re

title_lower = us_products['product_title'].str.lower().fillna('')
bullet_lower = us_products['product_bullet_point'].str.lower().fillna('')
brand_lower = us_products['product_brand'].str.lower().fillna('')

# --- keyword match (title only, non-capturing group to silence the warning) ---
pattern = r'\b(?:' + '|'.join(re.escape(k) for k in electronics_keywords) + r')\b'
keyword_mask = title_lower.str.contains(pattern, regex=True)

# --- brand match ---
brand_mask = brand_lower.isin(brand_allowlist)

# --- conglomerate-brand fix: brand alone isn't enough for brands that also sell
#     non-electronics goods (Samsung dryers, Sony movies/games/music, etc.) ---
conglomerate_brands = ['samsung', 'sony', 'lg', 'panasonic']

hardware_nouns = (
    r'\b(?:player|console|headphones?|speakers?|lens|camera|batter(?:y|ies)'
    r'|receiver|soundbar|walkman|radio|monitor|vcr|changer|memory card'
    r'|earbuds?|earphones?|turntable|amplifier)\b'
)
hardware_noun_mask = title_lower.str.contains(hardware_nouns, regex=True)

is_conglomerate = brand_lower.isin(conglomerate_brands)

# conglomerate brand-only matches must ALSO have a hardware noun to count;
# non-conglomerate brand matches are unaffected
brand_mask_final = brand_mask & ~(is_conglomerate & ~keyword_mask & ~hardware_noun_mask)

# --- exclusions: homonym collisions discovered across all samples so far ---
exclude_pattern = (
    r'\b(?:food processor'
    r'|dodge ram|ram \d{3,4}|ram trucks?|ram pickup|ram hitch'
    r'|dodge charger|charger challenger'
    r'|wood(?:en)? router|router table'
    r'|as seen on tv|tv show'
    r'|writing tablet|the emerald tablet)\b'
)
exclude_mask = title_lower.str.contains(exclude_pattern, regex=True)

# --- brand-only matches that are fiction books published under "Dell" ---
book_indicator = r'\b(?:a novel|: a novel|a memoir)\b'
brand_only_book_mask = (brand_mask & ~keyword_mask) & title_lower.str.contains(book_indicator, regex=True)

# --- final filter ---
electronics_products = us_products[
    (keyword_mask | brand_mask_final) & ~exclude_mask & ~brand_only_book_mask
].copy()

print(len(electronics_products))

# --- accessory tagging (not filtered out, just flagged for the methodology write-up) ---
accessory_terms = r'\b(?:case|cover|sleeve|bag|backpack|decal|sticker|skin|mount|holder|screen protector|charger cable|cable)\b'
electronics_products['is_accessory'] = electronics_products['product_title'].str.lower().str.contains(accessory_terms, regex=True)
print(electronics_products['is_accessory'].value_counts())

60874
is_accessory
False    38717
True     22157
Name: count, dtype: int64


In [17]:
electronics_products.sample(100, random_state=7)[['product_title','product_brand']]

,product_title,product_brand
1706612,"Hdiwousp HDMI Adapter for Wii Converter, Wii to Hdmi Converter Output Video Audio Adapter with 1M HDMI Cable Wii2HDMI with 3.5mm Audio Output Supports 720/1080P All Wii Display Modes for Nintendo",Hdiwousp
721381,"Lenovo ThinkPad P52 15.6” FHD IPS Anti-Glare Mobile Workstation, i7-8750H, NVIDIA Quadro P1000, 8GB DDR4, 1TB HDD, Webcam, BT, Backlit Keyboard, Fingerprint Reader, Thunderbolt 3, Windows 10 Pro",Lenovo
1318452,Polaroid One-Step 600 Instant Camera,Polaroid
256401,Bose Soundsport Free Wireless Charging Case (Black),Bose
1272946,"Huawei B315s-22 Unlocked 4G LTE 150 Mbps Mobile Wi-Fi Router (3G/4G LTE in Europe, Asia, Middle East, Africa) (Black)",HUAWEI
253816,Vintage Silver RechicGu Western Big Horn Sheep RAM Goat Skull Leather Rodeo Bolo Bola Tie Necktie,RechicGu
863291,"Qinroiot Home Security Camera , Wireless Rechargeable Battery Powered Outdoor&Indoor Wi-Fi Camera with 1080P Night Vision,PIR Motion Detection, Two-Way Audio, IP65 Waterproof Support Cloud/SD Slot",Qinroiot
1621520,"Bluetooth Speaker Portable Retro Radio, Trenbader.com Vintage Bluetooth Speaker for iPhone Samsung. Retro Speaker for Home Office Outdoor, Wooden Vintage Style, Mic",Yogyro
441926,iPhone 11 Pro Leather Case Slim Thin iPhone 11 Pro Case Leather Anti-Scratch Protective Phone Cover Cases for Apple iPhone 11 Pro 5.8” 2019 (Brown),Jijaogara
475412,"Progo Ultra Absorbent Microfiber Cleaning Cloths for LCD/LED TV, Laptop Computer Screen, iPhone, iPad and More. (6 Pack)",Progo


In [18]:
brand_only_matches = us_products[brand_mask & ~keyword_mask]
print(len(brand_only_matches))
brand_only_matches.sample(40, random_state=1)[['product_title','product_brand']]

6340


,product_title,product_brand
304558,"Corsair CRYSTAL 570X RGB Mid-Tower Case, 3 RGB Fans, Tempered Glass - White (CC-9011110-WW)",Corsair
492298,Logitech Multimedia Speaker System Z533 - Black,Logitech
230450,"Anker Alkaline AAA Batteries (24-Pack), Long-Lasting & Leak-Proof with PowerLock Technology, High Capacity Triple A Batteries with Adaptive Power and Superior Safety (Non-Rechargeable)",Anker
484393,LG K20 AT&T Unlocked 16 GB 4G LTE 5.3 inches 8 MP - Black (Renewed),LG
429463,"HP Envy 4520 Wireless All-in-One Photo Printer with Mobile Printing, Standard Ink Included, in Black (Renewed)",HP
1523607,Apple iPod Touch (32GB) - Product(Red) (Latest Model),Apple
766976,WD 4TB My Cloud Home Duo Personal Cloud Storage - WDBMUT0040JWT-NESN,Western Digital
1600180,"Apple iPad Air 2, 16GB, 4G + Wi-Fi - Gold (Renewed)",Apple
205418,Apple Silicone Case (for iPhone Xs Max) - White,Apple
821618,Sony SU-WL855 Ultra Slim Wall-Mount Bracket for Select Sony BRAVIA OLED and LED TVs,Sony


In [19]:
def find_trigger(idx):
    text = title_lower.loc[idx] + ' ' + bullet_lower.loc[idx]
    m = re.search(pattern, text)
    return m.group(0) if m else None

sample = electronics_products.sample(100, random_state=99)
sample = sample.copy()
sample['matched_keyword'] = [find_trigger(i) for i in sample.index]
sample[['product_title', 'product_brand', 'matched_keyword']]

,product_title,product_brand,matched_keyword
1718989,HP 24mh FHD Monitor - Computer Monitor with 23.8-Inch IPS Display (1080p) - Built-In Speakers and VESA Mounting - Height/Tilt Adjustment for Ergonomic Viewing - HDMI and DisplayPort - (1D0J9AA#ABA),HP,NaN
499019,SinLoon 2 in 1 Combine Mini PCI-E M.2 NGFF & mSATA SSD to SATA 3.0 Adapter Converter (SATA3.0/M),SinLoon,ssd
1537098,BLU G9 PRO | 2020 | All Day Battery | Unlocked | 6.3” HD+ Infinity Display | 128 GB | Triple 48MP Camera | US Version | US Warranty | Nightfall,BLU,camera
1372500,for 2003-07 Honda Accord Radio Android 10.1 Double Din Car Stereo Bluetooth Support Backup Camera Mirror Link Play 2G RAM 32G ROM,EZoneTronics,camera
169577,"ZOEAST(TM) AirPods Case,Thick Soft Portable Keychain Finger Ring Protective Silicone Skin Cover Case [Front LED Not Visible] Compatible with Apple AirPods Earphone Charging Case 2&1 (Blue Stitch)",Earphone,earphone
584164,"Bike Phone Mount - Phone Holder for Bike, Phone Stand for Motorcycle, Silicone Phone Holder for Handlebar fits iPhone, Samsung, Huawei, GPS, Adjustable Rotation Phone Mount for Bike, Stroller",AMALEN,phone
1213536,HONONJO AV SWITCHER 4 Way Converter Video Audio Selector for TV with Remote Control Video Switch Box （Electrical Volts: 110V）,HONONJO,tv
208607,"30PCS Anime Sexy Girl Laptop Stickers for Adult, Waterproof Decals Stickers for Water Bottle Laptop Skateboard Motorcycle Car Bike Luggage Trolley Case Decoration(Sexy Girl Stickers)",Awolf,laptop
392063,"Gigabyte AERO 15 OLED XA-7US5130SP 15"" Thin Bezel Samsung UHD Amoled, i7-9750H, NVIDIA GeForce RTX 2070, Samsung 16GB RAM, 512GB Intel 760P SSD, W10 Pro, RGB KB, 94W Ultra Slim Laptop",GIGABYTE,ram
1333889,SENNHEISER SET840 -TV RF Stereo TV Assistive Listening Headphones System,Sennheiser,tv


In [20]:
# did the publisher-Dell problem survive the title-only fix?
us_products[(us_products['product_brand'].str.lower() == 'dell') & 
            (brand_mask & ~keyword_mask)][['product_title','product_brand']].head(20)

,product_title,product_brand
170855,"Dell 1.92TB 12Gb/s 2.5"" SAS Solid State Drive Bundle with Tray, Compatible with Dell PowerEdge R640, R740, R740XD, R440, R540, R840, R940, T440, T640 Servers",Dell
174709,"Dell Inspiron 14 5400 2 in 1 - 14 Inch FHD Touchscreen, Intel Core i5, 12GB Memory, 256GB Solid State Drive, Windows 10 Home (Latest Model)- Titan Grey",Dell
179376,"Dell P2421D 24 Inch 1440p QHD, IPS Thin Bezel Monitor, HDMI, DisplayPort, VESA Certified, Black",Dell
179381,"Dell Curved Gaming Monitor 27 Inch Curved Monitor with 165Hz Refresh Rate, QHD (2560 x 1440) Display, Black - S2722DGM",Dell
179997,"Dell Dual Monitor Stand – MDS19, Black",Dell
185518,Dell U4919DW UltraSharp 49-inch QHD 5120x1440 32:9 Curved Monitor Bundle with Tech Smart USA Elite Suite 18 Standard Editing Software Bundle and 1 Year Extended Warranty,Dell
188967,"HGST 6TB SATA III 3.5-inch HDD | HUS726060ALE614 | Enterprise Internal Hard Disk Drive Bundle with Compatily Screwdriver | Compatible with Desktop Computers, Workstation PC, Mac, NAS & CCTV DVR",Dell
195062,Dell BIZLINK DisplayPort to DVI Dual Link Adapter USB Powered - XT625 / CN-0XT625,Dell
203110,"Alienware 25 Gaming Monitor - AW2518Hf, Full HD @ Native 240 Hz, 16:9, 1ms Response Time, DP, HDMI 2.0a, USB 3.0, AMD FreeSync, Tilt, Swivel, Height-Adjustable (Renewed)",Dell
203111,Alienware 25 Gaming Monitor - AW2518H (Renewed),Dell


In [21]:
conglomerates = ['samsung', 'lg', 'sony', 'panasonic']
conglomerate_brand_only = us_products[
    brand_lower.isin(conglomerates) & brand_mask & ~keyword_mask
]
print(len(conglomerate_brand_only))
conglomerate_brand_only.sample(min(40, len(conglomerate_brand_only)), random_state=3)[['product_title','product_brand']]

1933


,product_title,product_brand
349206,Sony Universal Remote Control,Sony
1677699,Sony ICF-P36 Portable AM/FM Radio - Black,Sony
855715,SONY X800 2K/4K UHD - 2D/3D - Wi-Fi 2.4/5.0 Ghz - Clear Audio - Multi System All Region Blu Ray Disc DVD Player 100-240V 50/60Hz Auto,Sony
484681,LG Tone Ultra HBS-835 Bluetooth Stereo Headset - Wireless with JBL Signature Sound (Black),LG
1718624,"LG NanoCell 80 Series 65” Alexa Built-in 4k Smart TV (3840 x 2160), 60Hz Refresh Rate, AI-Powered 4K Ultra HD, Dolby Cinema, Dolby Vision (65NANO80UPA, 2021)",LG
1301610,Spider-Man: Homecoming [Blu-ray],Sony
484475,LG LP1017WSR 115V Portable Air Conditioner with Remote Control in White for Rooms up to 250-Sq. Ft. (Renewed),LG
606736,"Panasonic Water Jet Nozzle Replacement, Compatible with EW-DJ10-W, EW-DJ10-A Dental Flossers, White, 2 Count",Panasonic
809688,"LG WM3488HW 24"" Washer/Dryer Combo with 2.3 cu. ft. Capacity, Stainless Steel Drum in White",LG
679745,Sony NW-ZX300 Hi-Res Walkman 64GB Digital Music Player (Black),Sony


In [22]:
sony_brand_only = us_products[(brand_lower == 'sony') & brand_mask & ~keyword_mask]
print(len(sony_brand_only))
sony_brand_only.sample(min(40, len(sony_brand_only)), random_state=4)[['product_title','product_brand']]

705


,product_title,product_brand
426975,Horizon Zero Dawn: Complete Edition - PlayStation 4 Digital Download Code,Sony
755213,Sony 12T120VR 120-Minute Premium VHS Cassettes (12-Pack) (Discontinued by Manufacturer),Sony
1515915,Sony UBP-X800M2 4K Ultra HD Blu-ray Player with HDR with 2-6ft. High Speed Premium HDMI Cables and 14ft Ethernet Cable Bundle,Sony
1717646,Sony SEL35F14Z Distagon T FE 35mm f/1.4 ZA Standard-Prime Lens for Mirrorless Cameras,Sony
679588,Sony HAPZ1ES 1TB Hi-Res Music Player System,Sony
819615,Sony SELP1650 16-50mm Power Zoom Lens,Sony
542297,Sony STR-DN1080 7.2-Channel Home Theater AV Receiver Bundled with Active Subwoofer and Seven Sony Speakers (9 Items),Sony
825604,Days Gone PS4 Playstation 4,Sony
654812,The ICO and Shadow of the Colossus Collection (Renewed),Sony
679769,"Sony XBR85X950H 85"" X950H 4K Ultra HD LED TV (2020) with Deco Gear Soundbar Bundle",Sony


In [23]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

validation_sample = electronics_products.sample(200, random_state=100)[['product_title', 'product_brand', 'is_accessory']]
validation_sample.to_csv('electronics_filter_validation_sample.csv', index=False)
validation_sample

,product_title,product_brand,is_accessory
1243179,"Anyshock Desk Mat, Extended Gaming Mouse Pad Large 31.5"" x 15.7"" XL Keyboard pad with Stitched Edges Non Slip Base, Water-Resistant Computer Laptop Desk Pad for Office and Home(White Marble)",Anyshock,False
1600447,"Matone Tablet Holder with Strongest Large Suction Cup, Universal Tablet Stand Phone Holder for iPhone iPad Series/Samsung Galaxy Tabs/Fire HD and More, Great for Mirrors/Windows/Walls",Matone,True
443564,Headphone Adapter,Jafuam,False
172689,"CZ Portable 5V/5.1A 4-USB Port Smart Quick Charge Travel Charger with Holder Function for Smartphones & Tablets & Power Bank & Bluetooth Headset, 100-240V Wide Voltage, US Plug (Color : Black)",Nologo,True
217653,AOO Keyboard Cleaner Gel Dust Cleaning Mud Laptop Keyboards PC Tablet Screen Interior Car Detailing Kit (5-Pack(14.10oz/400g)),AOO,False
442436,"Apple iPhone 6S, 64GB, Space Gray - For AT&T / T-Mobile (Renewed)",Apple,False
464967,"8.0 inch Tablet Case, Mptie Universal Protective PU Leather Case Cover for All 7.5-8.5 inch iPad Mini 1 2 3 4 5,Galaxy Tab A 8.0/Tab E 8.0, F ire HD 8 and More (Books)",Motie,True
1597850,TIKRO iPhone Charger [Apple MFi Certified] 5 Pack - 6FT Lightning Cable Nylon Woven High Speed Data Sync Cord Compatible iPhone 12 11 Pro Max X 8 7 6S Plus SE - Color,TIKRO,True
179978,"ASUS VE278H 27"" Full HD 1920x1080 2ms HDMI VGA Back-lit LED Monitor",ASUS,False
1212956,Asus VG248QG 24” G-Sync Compatible Gaming Monitor 165Hz Full HD 1080P 0.5ms DP HDMI DVI Eye Care (Renewed),ASUS,False


In [6]:
electronics_ids = set(electronics_products['product_id'])
electronics_examples = shopping_queries_dataset_examples[
    (shopping_queries_dataset_examples['product_locale'] == 'us') &
    (shopping_queries_dataset_examples['product_id'].isin(electronics_ids))
]
print(electronics_examples['query'].nunique(), "distinct electronics queries")
print(len(electronics_examples), "electronics query-product judgment rows")
print(electronics_examples['esci_label'].value_counts())

NameError: name 'electronics_products' is not defined

In [25]:
electronics_examples['query_length'] = electronics_examples['query'].str.split().str.len()
unique_queries = electronics_examples.drop_duplicates(subset='query')[['query', 'query_length']]
print(unique_queries['query_length'].describe())

count    17998.000000
mean         3.612401
std          1.958622
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         29.000000
Name: query_length, dtype: float64


In [31]:
unique_queries = electronics_examples.drop_duplicates(subset='query')[['query', 'query_length']]

print("=== 1-2 words ===")
print(unique_queries[unique_queries['query_length'] <= 2].sample(100, random_state=1)['query'].to_list())

print("\n=== 3-4 words ===")
print(unique_queries[unique_queries['query_length'].between(3, 4)].sample(100, random_state=1)['query'].to_list())

print("\n=== 5-6 words ===")
print(unique_queries[unique_queries['query_length'].between(5, 6)].sample(100, random_state=1)['query'].to_list())

print("\n=== 7+ words ===")
print(unique_queries[unique_queries['query_length'] >= 7].sample(100, random_state=1)['query'].to_list())

=== 1-2 words ===
['small rig', 'mmcx cable', 'desktop pc', 'motorcycle', 'lg ud58', 'fgteev book', 'ips curved', 'cr123a 3v', 'canon t7i', 'pink wristlet', 'princess toys', 'earbud', 'gadget guard', 'inazuma gw250', 'plan b', 'assassins creed', 'motorola', 'usb hdmi', 'kindle', 'baywatch pop', 'rolling backpack', 'colageno capsules', 'baofeng radio', 'sonos subwoofer', 'snow shovel', 'blu', 'dji ronin', 'ps4 fan', 'printing calculators', 'digital frame', 'aelfox', 'pbt98 pyle', 'side boards/buffet', 'victsing', 'e katerina', 'giftcards', 'black pods', 'motorolaone', 'mi phone', 'fibroplex 240', 'bape', 'wood engravers', 'labradoodle decal', 'bts', 'bushnell wingman', 'stephanie bower', 'spectre', 'logitech', 'hidden storage', 'ota tuner', 'gsm', 'summicron-m 35mm', 'stacked chests', 'xiaomi watch', 'cd/dvd printers', 'billie eilish', 'bride hanger', 'horns', 'makeup eraser', 'hide wire', 'sports bag', 'snaptain', 'samsung buds', 'nintendo+switch+games+dead cells', 'magic mount', 'popc

In [5]:
electronics_examples_with_source = electronics_examples.merge(
    shopping_queries_dataset_sources, on='query_id', how='left'
)
print(electronics_examples_with_source['source'].value_counts())

NameError: name 'electronics_examples' is not defined

In [33]:
# quick eyeball before deciding
unique_queries_check = electronics_examples_with_source.drop_duplicates(subset='query_id')
print(unique_queries_check[unique_queries_check['source']=='parse_pattern']['query'].sample(10, random_state=1).to_list())
print(unique_queries_check[unique_queries_check['source']=='nlqec']['query'].sample(10, random_state=1).to_list())

['bose lifestyle', 'xbox one final fantasy x', 'star plug in night light', 'ideapad slim 7', 'alexa cube remote control', 'noristudio sticker', 'pound fitness sticks', 'yellow and grey wall decor', 'xlr splitter 2 female 1 male', 'mobile printers for travel']
['The laptop should be a windows laptop. Light with powerful processor and good memory ', 'One with an SSD drive, good RAM, fast processor, and a 15 screen, lightweight but ideally with a built in DVD/CD drive', 'Portable, touch-screen, with a pen, high resolution display, good speakers. ', 'One that’s fast, reliable and has a good processor. I’d ideally like it to be light-weight too so that I can easily transport it.', 'Thin, lightweight but also waterproof and looks good. Hood not essential. ', 'I would buy the exact same laptop that I have now. I would get another Apple Macbook Air.', 'Not sure. Maybe a a standard size (15 or 17??) screen middle of the range laptop that is either black or purple. It might be a Dell or Acer ', 

In [34]:
training_pool_sources = ['other']  # extend with 'parse_pattern'/'nlqec' if the check above shows they're normal queries

training_pool = electronics_examples_with_source[
    electronics_examples_with_source['source'].isin(training_pool_sources)
].drop_duplicates(subset='query_id')[['query_id', 'query']]

training_pool['query_length'] = training_pool['query'].str.split().str.len()
print(len(training_pool), "candidate queries for the 500")
print(training_pool['query_length'].describe())

16418 candidate queries for the 500
count    16418.000000
mean         3.434219
std          1.481515
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         12.000000
Name: query_length, dtype: float64


In [36]:
training_pool_sources = ['other', 'parse_pattern']

training_pool = electronics_examples_with_source[
    electronics_examples_with_source['source'].isin(training_pool_sources)
].drop_duplicates(subset='query_id')[['query_id', 'query']]

training_pool['query_length'] = training_pool['query'].str.split().str.len()
print(len(training_pool), "candidate queries for the 500")
print(training_pool['query_length'].describe())

17058 candidate queries for the 500
count    17058.000000
mean         3.455153
std          1.486453
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         12.000000
Name: query_length, dtype: float64


In [37]:
def bucket(n):
    if n <= 2:
        return '1-2'
    elif n <= 4:
        return '3-4'
    elif n <= 6:
        return '5-6'
    else:
        return '7+'

training_pool['length_bucket'] = training_pool['query_length'].apply(bucket)
bucket_counts = training_pool['length_bucket'].value_counts()
print(bucket_counts)
print((bucket_counts / len(training_pool) * 100).round(1))

length_bucket
3-4    8579
1-2    4804
5-6    3086
7+      589
Name: count, dtype: int64
length_bucket
3-4    50.3
1-2    28.2
5-6    18.1
7+      3.5
Name: count, dtype: float64


In [38]:
sample_sizes = {'1-2': 90, '3-4': 230, '5-6': 110, '7+': 70}

samples = []
for bucket_name, n in sample_sizes.items():
    bucket_pool = training_pool[training_pool['length_bucket'] == bucket_name]
    samples.append(bucket_pool.sample(n, random_state=42))

labeling_set = pd.concat(samples).reset_index(drop=True)
print(len(labeling_set))
print(labeling_set['length_bucket'].value_counts())

labeling_set.to_csv('classifier_labeling_set_500.csv', index=False)

500
length_bucket
3-4    230
5-6    110
1-2     90
7+      70
Name: count, dtype: int64


In [39]:
suspect_queries = ['belt', 'pa1', "amazoncom'", 'mail baskets', 'can crusher', 
                    'pokemon egg', 'hookah flavors', 'lockpicks kit', 
                    'lawn trencher', "mar'villa"]

for q in suspect_queries:
    match = training_pool[training_pool['query'] == q]
    if match.empty:
        continue
    qid = match['query_id'].iloc[0]
    rows = electronics_examples_with_source[electronics_examples_with_source['query_id'] == qid]
    merged = rows.merge(electronics_products[['product_id', 'product_title']], on='product_id', how='left')
    print(f"\n=== {q!r} ===")
    print(merged[['esci_label', 'product_title']].to_string(index=False))


=== 'belt' ===
esci_label                                                                                                                                    product_title
         E G-Run Hydration Running Belt with Bottles - Water Belts for Woman and Men - iPhone Belt for Any Phone Size - Fuel Marathon Race Pack for Runners

=== 'pa1' ===
esci_label                                                              product_title
         E               XP-PEN PA1 Battery-Free Stylus only Deco Pro Graphics Tablet
         E                    SNOM PA1 Public Address IP Phone Paging System (Latest)
         I Snom SNO-D715 Professional Sip Desk Telephone Voip Phone and Device, Black

=== "amazoncom'" ===
esci_label                                          product_title
         I              Apple AirPods with Wireless Charging Case
         E App Store & iTunes Surprise Gift Card- E-mail Delivery

=== 'mail baskets' ===
esci_label                                                           

In [40]:
# a query only counts as "electronics" if it has at least one Exact or Substitute
# labeled product within the electronics catalog — not just any judged product
positive_labels = ['E', 'S']

electronics_query_ids = electronics_examples_with_source[
    electronics_examples_with_source['esci_label'].isin(positive_labels)
]['query_id'].unique()

training_pool_v2 = electronics_examples_with_source[
    (electronics_examples_with_source['query_id'].isin(electronics_query_ids)) &
    (electronics_examples_with_source['source'].isin(['other', 'parse_pattern']))
].drop_duplicates(subset='query_id')[['query_id', 'query']].copy()

training_pool_v2['query_length'] = training_pool_v2['query'].str.split().str.len()
print(len(training_pool_v2))

15622


In [41]:
# 1. "ram" as a standalone word needs to require memory-context, not just any brand containing "Ram"
ram_context_pattern = r'\b\d+\s?(?:gb|tb|mb)\s*ram\b|\bram\s*\d+\s?(?:gb|tb|mb)\b|\bddr[345]\b'
# replace bare "ram" in electronics_keywords with this context-aware check instead

# 2. cordless outdoor power tools — a recurring leak across brands (Greenworks, Ryobi, MOCOHANA)
outdoor_tool_pattern = r'\b(?:edger|trimmer|chainsaw|lawn\s?mower|leaf\s?blower|hedge\s?trimmer|pole\s?saw|weed\s?wacker)\b'
outdoor_tool_mask = title_lower.str.contains(outdoor_tool_pattern, regex=True)

electronics_products = electronics_products[~outdoor_tool_mask].copy()
print(len(electronics_products))

C:\Users\chamo\AppData\Local\Temp\ipykernel_27072\3775437455.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  electronics_products = electronics_products[~outdoor_tool_mask].copy()


60664


In [42]:
electronics_ids = set(electronics_products['product_id'])
electronics_examples = shopping_queries_dataset_examples[
    (shopping_queries_dataset_examples['product_locale'] == 'us') &
    (shopping_queries_dataset_examples['product_id'].isin(electronics_ids))
]
print(electronics_examples['query'].nunique(), "distinct electronics queries")
print(len(electronics_examples), "electronics query-product judgment rows")
print(electronics_examples['esci_label'].value_counts())

17919 distinct electronics queries
122004 electronics query-product judgment rows
esci_label
E    83948
S    25595
I     8502
C     3959
Name: count, dtype: int64


In [43]:
electronics_examples_with_source = electronics_examples.merge(
    shopping_queries_dataset_sources, on='query_id', how='left'
)
print(electronics_examples_with_source['source'].value_counts())

source
other            112988
parse_pattern      3478
negations          3379
behavioral         1170
nlqec               989
Name: count, dtype: int64


In [44]:
positive_labels = ['E', 'S']

electronics_query_ids = electronics_examples_with_source[
    electronics_examples_with_source['esci_label'].isin(positive_labels)
]['query_id'].unique()

training_pool_v2 = electronics_examples_with_source[
    (electronics_examples_with_source['query_id'].isin(electronics_query_ids)) &
    (electronics_examples_with_source['source'].isin(['other', 'parse_pattern']))
].drop_duplicates(subset='query_id')[['query_id', 'query']].copy()

training_pool_v2['query_length'] = training_pool_v2['query'].str.split().str.len()
print(len(training_pool_v2))

15550


In [45]:
training_pool_v2['length_bucket'] = training_pool_v2['query_length'].apply(bucket)
print(training_pool_v2['length_bucket'].value_counts())

length_bucket
3-4    7829
1-2    4344
5-6    2839
7+      538
Name: count, dtype: int64


In [46]:
sample_sizes = {'1-2': 90, '3-4': 230, '5-6': 110, '7+': 70}

samples = []
for bucket_name, n in sample_sizes.items():
    bucket_pool = training_pool_v2[training_pool_v2['length_bucket'] == bucket_name]
    samples.append(bucket_pool.sample(n, random_state=42))

labeling_set_v2 = pd.concat(samples).reset_index(drop=True)
print(len(labeling_set_v2))
print(labeling_set_v2['length_bucket'].value_counts())

labeling_set_v2.to_csv('classifier_labeling_set_500_v2.csv', index=False)

500
length_bucket
3-4    230
5-6    110
1-2     90
7+      70
Name: count, dtype: int64
